# Environment

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

on_drive = False

CLUSTER_DATASET = 'dataset_name'
CLUSTER_EXPERIMENT_FAMILY = 'screen'
CLUSTER_L_H_SETTING = '168_24'
CLUSTER_MODEL_NAME = 'chronos2'
CLUSTER_SPACE, CLUSTER_METRIC, CLUSTER_K, CLUSTER_RETRIEVAL_MODE = 'raw', 'euclidean', 1, 'online'
CLUSTER_EXTRACTION_RUN = 'run_0'
CLUSTER_EXTRACTION_DIR = Path('outputs/extraction') / CLUSTER_DATASET / CLUSTER_L_H_SETTING / CLUSTER_MODEL_NAME / CLUSTER_SPACE / CLUSTER_METRIC / str(CLUSTER_K) / CLUSTER_RETRIEVAL_MODE / CLUSTER_EXTRACTION_RUN
CLUSTER_RESULT_SPECS = [('cov_ridge_shared', 'run_0'), ('bayes_cov_shared', 'run_0')]
CLUSTER_RESULT_DIRS = [Path('outputs/adaptation') / CLUSTER_EXPERIMENT_FAMILY / CLUSTER_DATASET / CLUSTER_L_H_SETTING / CLUSTER_MODEL_NAME / formula / CLUSTER_SPACE / CLUSTER_METRIC / str(CLUSTER_K) / CLUSTER_RETRIEVAL_MODE / run for formula, run in CLUSTER_RESULT_SPECS]
COLAB_EXTRACTION_DIR = '/content/drive/MyDrive/path/to/outputs/extraction/dataset/168_24/chronos2/raw/euclidean/1/online/run_0'
COLAB_RESULT_DIRS = ['/content/drive/MyDrive/path/to/outputs/adaptation/screen/dataset/168_24/chronos2/cov_ridge_shared/raw/euclidean/1/online/run_0']
COLAB_REPO_DIR = '/content/drive/MyDrive/path/to/ts-ifa'
INSTALL_PROJECT_ON_COLAB = True
INSTALL_WIDGETS_IF_MISSING = True

if on_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path('/content/drive/MyDrive/Recherche/Thèse Gaspard/Datasets')
    %cd $COLAB_REPO_DIR
    %cd src
    if INSTALL_PROJECT_ON_COLAB:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            '--no-deps', '-e', COLAB_REPO_DIR,
        ])
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets>=8.1',
        ])
else:
    project_src = Path.cwd().parent if Path.cwd().name == 'visu' else Path.cwd() / 'src'
    if not project_src.is_dir():
        project_src = Path.cwd()
    %cd $project_src
    data_path = Path('../datasets')
if not on_drive and INSTALL_WIDGETS_IF_MISSING and importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', 'ipywidgets>=8.1',
    ])

EXTRACTION_DIR = COLAB_EXTRACTION_DIR if on_drive else Path('..') / CLUSTER_EXTRACTION_DIR
RESULT_DIRS = [Path(path) for path in COLAB_RESULT_DIRS] if on_drive else [Path('..') / path for path in CLUSTER_RESULT_DIRS]
EXTRACTION_DIR, RESULT_DIRS

# Imports

In [ ]:
from IPython.display import display

from src.visu.dashboard import (
    baseline_section,
    data_summary,
    gates_section,
    horizon_section,
    load_dashboard_data,
    query_section,
    ts_ifa_section,
    window_scatter_section,
)

# Data loading

In [ ]:
data = load_dashboard_data(EXTRACTION_DIR, RESULT_DIRS)
print(data_summary(data))

# Query and retrieved examples

For a query origin $s$, the solid segment is the observed lookback and the dashed segment is its future:

$$X_s=(z_{s-L+1},\ldots,z_s),\qquad Y_s=(z_{s+1},\ldots,z_{s+H}).$$

Each retrieved neighbour $r_j$ is drawn in the same way as $(X_{r_j},Y_{r_j})$.

In [ ]:
display(query_section(data))

# Window-wise diagnostics

For prediction $\hat y'$ and loss $\ell$, each point uses one query window $i$:

$$m_i(\hat y')=\frac1H\sum_{h=1}^{H}\ell(y_{i,h},\hat y'_{i,h}).$$

The y-axis is respectively $m_i(\hat y')$, $m_i(\hat y')-m_i(\hat y'')$, or $(m_i(\hat y')-m_i(\hat y''))/m_i(\hat y'')$. The log buttons use a symmetric-log scale so signed values remain visible.

In [ ]:
display(window_scatter_section(data))

# Horizon-wise diagnostics

For horizon $h$, aggregation is performed across the $N$ query windows:

$$A_h(\hat y')=\frac1N\sum_{i=1}^{N}\ell(y_{i,h},\hat y'_{i,h}).$$

Direct view plots $A_h(\hat y')$; improvement plots $A_h(\hat y')-A_h(\hat y'')$; relative view plots $(A_h(\hat y')-A_h(\hat y''))/A_h(\hat y'')$. Here $\ell_{\mathrm{MSE}}=(\hat y-y)^2$, $\ell_{\mathrm{nMSE}}=((\hat y-y)/\operatorname{std}(X))^2$, and difference is $\hat y-y$.

In [ ]:
display(horizon_section(data))

# Baseline coefficient importance

For a fitted ridge baseline, feature importance is $I_j=|\beta_j|$ in shared mode and $I_j=H^{-1}\sum_h|\beta_{h,j}|$ in horizon mode. Convex mixtures use the absolute mixing coefficient.

In [ ]:
display(baseline_section(data))

# Gates

At threshold $\tau$, a score $q$ selects the retrieval-informed candidate $c$ through

$$g_\tau=\mathbf 1[q>\tau],\qquad \hat y_\tau=g_\tau\hat y_c+(1-g_\tau)\hat y_{\mathrm{vanilla}}.$$

The threshold plots report routing accuracy, $\mathrm{TPR}=\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$, nMSE, and $100(\mathrm{nMSE}_{\mathrm{vanilla}}-\mathrm{nMSE}_{\tau})/\mathrm{nMSE}_{\mathrm{vanilla}}$.

In [ ]:
display(gates_section(data))

# TS-IFA rooter coefficients

TS-IFA corrects the frozen forecast using candidate residuals $\Delta_{i,c,h}$:

$$\hat y_{i,h}=\hat y_{i,h}^{\mathrm{vanilla}}+\sum_c\alpha_{i,c,h}\Delta_{i,c,h}.$$

The heatmap shows $\alpha_{c,h}$ for the fixed ridge rooter or $N^{-1}\sum_i\alpha_{i,c,h}$ for the neural rooter.

In [ ]:
display(ts_ifa_section(data))